# Data cleaning

In [57]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport


## 1. Importing data

In [58]:
csv_path = "../data/raw/store-data-6aa6d7a3f171f140353680.csv"

df = pd.read_csv(csv_path)

## 2. Fixing text

In [59]:
str_cols = df.select_dtypes(include="object").columns
df[str_cols] = df[str_cols].apply(lambda s: s.str.strip())

df["Segment"] = df["Segment"].replace({"Consumerr": "Consumer"})
df["Segment"] = df["Segment"].replace({"Home Ofice": "Home Office"})
df["Segment"] = df["Segment"].replace({"Corporrate": "Corporate"})


df["Customer Name"] = df["Customer Name"].str.strip().str.title()
df["Category"] = df["Category"].str.strip().str.title()
df["Segment"] = df["Segment"].str.strip().str.title()
df["State"] = df["State"].str.strip().str.title()
df["City"] = df["City"].str.strip().str.title()


## 3.Removing duplicates

In [60]:
df = df.drop_duplicates()

## 4. Fixing invalid dates

In [61]:
for column in ['Order Date', 'Ship Date']:
    if column in df.columns:
        df[column] = pd.to_datetime(df[column], errors='coerce')

if {'Order Date', 'Ship Date'}.issubset(df.columns):
    invalid_dates = df[
        df['Order Date'].notna() &
        df['Ship Date'].notna() &
        (df['Ship Date'] < df['Order Date'])
    ]
    df = df.drop(invalid_dates.index)

## 5. Filling missing data

In [63]:
df['Customer Name'] = df['Customer Name'].fillna(
    df.groupby('Customer ID')['Customer Name'].transform('first')
)

df['Postal Code'] = df['Postal Code'].fillna(
    df.groupby('City')['Postal Code'].transform('first')
)



df["Shipping Days"] = (df["Ship Date"] - df["Order Date"]).dt.days

avg_days = df.groupby("Ship Mode")["Shipping Days"].median()

display(avg_days)

days_to_add = pd.to_timedelta(df["Ship Mode"].map(avg_days))

df["Ship Date"] = np.where(df["Ship Date"] < df["Order Date"],df["Order Date"] + days_to_add, df["Ship Date"])


df["Shipping Days"] = (df["Ship Date"] - df["Order Date"]).dt.days

df["Ship Date"] = np.where(df["Shipping Days"] > 200, df["Order Date"] + days_to_add, df["Ship Date"])


df["Quantity"] = df.groupby(["Product ID", "Profit", "Discount"])["Quantity"].ffill().bfill()
df["Sales"] = df.groupby(["Product ID", "Profit", "Discount"])["Sales"].ffill().bfill()


df["Discount"] = df.groupby(["Product ID", "Profit", "Sales"])["Discount"].transform(lambda x: x.median())
df["Quantity"] = df.groupby(["Product ID", "Profit", "Sales"])["Quantity"].transform(lambda x: x.median())
df["Sales"] = df.groupby(["Product ID", "Profit", "Quantity"])["Sales"].transform(lambda x: x.median())


display(df["Discount"])
display(df[df["Quantity"] < 0])




Ship Mode
First Class       2.0
Same Day          0.0
Second Class      3.0
Standard Class    5.0
Name: Shipping Days, dtype: float64

0        0.00
1        0.00
2        0.00
3        0.45
4        0.20
         ... 
10034    0.20
10045    0.00
10046    0.00
10052    0.00
10062    0.20
Name: Discount, Length: 9997, dtype: float64

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Shipping Days
439,440,CA-2017-157252,2017-01-20,2017-01-23,Second Class,CV-12805,Cynthia Voltz,Corporate,United States,New York City,...,East,FUR-CH-10003396,Furniture,Chairs,Global Deluxe Steno Chair,207.846,-1.0,0.1,2.3094,3.0
650,651,CA-2016-160745,2016-12-11,2016-12-16,Second Class,AR-10825,Anthony Rawles,Corporate,United States,Vancouver,...,West,TEC-AC-10001142,Technology,Accessories,First Data FD10 PIN Pad,316.000,-0.5,0.0,31.6000,5.0
1481,1482,US-2016-108455,2016-12-02,2016-12-08,Standard Class,MK-18160,Mike Kennedy,Consumer,United States,San Francisco,...,West,OFF-ST-10002214,Office Supplies,Storage,X-Rack File for Hanging Folders,33.870,-5.0,0.0,8.8062,6.0
2302,2303,CA-2016-106558,2016-01-17,2016-01-21,Standard Class,DL-13495,Dionis Lloyd,Corporate,United States,Columbus,...,South,TEC-AC-10001142,Technology,Accessories,First Data FD10 PIN Pad,316.000,-0.5,0.0,31.6000,4.0
6412,6413,CA-2017-151211,2017-08-17,2017-08-23,Standard Class,AH-10120,Adrian Hane,Home Office,United States,Louisville,...,South,OFF-BI-10002735,Office Supplies,Binders,GBC Prestige Therm-A-Bind Covers,102.930,-1.0,0.0,48.3771,6.0
6511,6512,CA-2017-167640,2017-03-06,2017-03-10,Standard Class,FC-14245,Frank Carlisle,Home Office,United States,San Francisco,...,West,OFF-AR-10003158,Office Supplies,Art,Fluorescent Highlighters by Dixon,23.880,-5.0,0.0,8.1192,4.0
7419,7420,US-2016-114013,2016-03-13,2016-03-15,Second Class,SC-20770,Stewart Carmichael,Corporate,United States,Philadelphia,...,East,OFF-ST-10002574,Office Supplies,Storage,"SAFCO Commercial Wire Shelving, Black",552.560,-5.0,0.2,-138.1400,2.0
7799,7800,CA-2017-166184,2017-03-24,2017-03-27,First Class,HR-14830,Harold Ryan,Corporate,United States,New York City,...,East,FUR-CH-10003396,Furniture,Chairs,Global Deluxe Steno Chair,207.846,-1.0,0.1,2.3094,3.0
7894,7895,CA-2017-124744,2017-06-21,2017-06-25,Standard Class,EH-14125,Eugene Hildebrand,Home Office,United States,Wheeling,...,East,OFF-BI-10002852,Office Supplies,Binders,Ibico Standard Transparent Covers,82.400,-5.0,0.0,40.3760,4.0
8730,8731,CA-2015-103870,2015-12-27,2015-12-31,Standard Class,SP-20860,Sung Pak,Corporate,United States,Murfreesboro,...,South,TEC-AC-10004227,Technology,Accessories,SanDisk Ultra 16 GB MicroSDHC Class 10 Memory ...,72.744,-5.0,0.2,-12.7302,4.0


In [ ]:
df = df.drop_duplicates()

In [ ]:

profile = ProfileReport(df, title="Profiling Report after cleaning")
profile.to_file("../reports/report_cleaning.html")

